
# Unit 05 — File


It keeps the same learning style as the earlier units:
- summary of core ideas
- readable examples
- common mistakes
- worked problems
- practice problems
- reference solutions
- self-check assertions

This version uses **clean file handling** with `with open(...) as f:` whenever we read or write files.




## 0. Learning goals

By the end of this unit, you should be able to:

1. open a text file for reading
2. read file contents line by line
3. understand the difference between `for line in f`, `readline()`, and `readlines()`
4. remove trailing newline characters when needed
5. split structured text into fields
6. open a text file for writing
7. remember that `write()` does **not** add `'\n'` automatically
8. solve file-processing problems cleanly
9. compare two files
10. extract information from tagged text such as `<headline> ... </headline>`



## 1. Unit summary

### Reading files
A common pattern is:

```python
with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        ...
```

Important notes:
- each `line` often ends with `'\n'`
- use `line.strip()` when you want to remove outer whitespace and newline
- use `line.rstrip('\n')` when you want to remove only the final newline

### Writing files
A common pattern is:

```python
with open(filename, "w", encoding="utf-8") as out:
    out.write(text)
```

Important note:
- `write()` does **not** add a newline for you  
- if you want a new line, write `out.write(text + '\n')`

### Common file-reading tools
- `for line in f:` → best for reading all lines one by one
- `f.readline()` → read one line at a time manually
- `f.readlines()` → read the whole file into a list of strings

### Structured text
If each line contains fields, you often do:

```python
parts = line.strip().split(":")
```

or

```python
parts = line.split()
```

depending on the file format.


In [1]:

# Unit 05 demo file setup
from pathlib import Path

BASE = Path("unit05_demo_files")
BASE.mkdir(exist_ok=True)

(BASE / "simple_lines.txt").write_text(
    "line1\nline2\nline3\n",
    encoding="utf-8"
)

(BASE / "mixed_blank_lines.txt").write_text(
    "alpha\n\n   \nbeta\ngamma\n",
    encoding="utf-8"
)

(BASE / "news.txt").write_text(
    "<headline>Rain expected tomorrow</headline>\n"
    "No headline on this line\n"
    "<headline>Stocks rise after report</headline>\n"
    "<headline>Python class cancelled</headline>\n",
    encoding="utf-8"
)

(BASE / "same1.txt").write_text(
    "A\nB\nC\n",
    encoding="utf-8"
)
(BASE / "same2.txt").write_text(
    "A\nB\nC\n",
    encoding="utf-8"
)
(BASE / "diff.txt").write_text(
    "A\nB\nX\n",
    encoding="utf-8"
)

(BASE / "data.txt").write_text(
    "5913842721:Somsak Rakrian:1:56.6\n"
    "5913845921:Somsri Deeying:2:78.0\n"
    "5913856821:Rakchard Yingcheep:2:89.0\n"
    "5913861321:Thumdee Tong Daidee:2:99\n"
    "591387721:Somrak Rakrian:10:84.25\n",
    encoding="utf-8"
)

(BASE / "answers.txt").write_text(
    "ABCDAB\n"
    "65010001 ABCDAB\n"
    "65010002 ABCD  \n"
    "65010003 AXCDAB\n",
    encoding="utf-8"
)

print("Created demo files in:", BASE.resolve())
for p in sorted(BASE.iterdir()):
    print("-", p.name)


Created demo files in: /Users/ppathorn/Downloads/unit05_demo_files
- answers.txt
- data.txt
- diff.txt
- mixed_blank_lines.txt
- news.txt
- same1.txt
- same2.txt
- simple_lines.txt



## 2. Clean reading patterns

### 2.1 Read line by line

This is the most important pattern in this chapter.


In [2]:

with open(BASE / "simple_lines.txt", "r", encoding="utf-8") as f:
    for line in f:
        print(repr(line))


'line1\n'
'line2\n'
'line3\n'



Notice that each line still has `'\n'` at the end.

When you want the text without the final newline, use `rstrip('\n')` or `strip()` depending on your goal.


In [3]:

with open(BASE / "simple_lines.txt", "r", encoding="utf-8") as f:
    for line in f:
        print(line.rstrip("\n"))


line1
line2
line3



### 2.2 `readline()` vs `readlines()`

- `readline()` reads **one** line
- `readlines()` reads the **whole** file into a list


In [4]:

with open(BASE / "simple_lines.txt", "r", encoding="utf-8") as f:
    first_line = f.readline()
    second_line = f.readline()
    print("first_line =", repr(first_line))
    print("second_line =", repr(second_line))

with open(BASE / "simple_lines.txt", "r", encoding="utf-8") as f:
    all_lines = f.readlines()
    print("all_lines =", all_lines)


first_line = 'line1\n'
second_line = 'line2\n'
all_lines = ['line1\n', 'line2\n', 'line3\n']



## 3. Clean writing patterns

### 3.1 Write text to a file

Remember: `write()` does **not** add a newline automatically.


In [5]:

out_path = BASE / "written_example.txt"

with open(out_path, "w", encoding="utf-8") as out:
    out.write("hello")
    out.write("\n")
    out.write("world")

print(out_path.read_text(encoding="utf-8"))


hello
world



## 4. Common mistakes

### Mistake 1: forgetting to remove `'\n'`

Suppose you search for a name inside a file.


In [6]:

(BASE / "names.txt").write_text("Ann\nBob\nChris\n", encoding="utf-8")

target = "Bob"

found_wrong = False
with open(BASE / "names.txt", "r", encoding="utf-8") as f:
    for line in f:
        if line == target:   # wrong
            found_wrong = True
            break

found_right = False
with open(BASE / "names.txt", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip() == target:   # right
            found_right = True
            break

print("wrong comparison:", found_wrong)
print("right comparison:", found_right)


wrong comparison: False
right comparison: True



### Mistake 2: path strings with backslashes

A path like `'c:\temp\data.txt'` can accidentally contain escape sequences such as `\t`.

Safer options:
- use forward slashes: `'c:/temp/data.txt'`
- or escape backslashes: `'c:\\temp\\data.txt'`

In this notebook we use `pathlib.Path`, which is even cleaner.



### Mistake 3: opening a missing file for reading

This can raise an error. In grader-style problems, the file is usually guaranteed to exist.  
In real code, it is better to handle the case carefully.


In [7]:

missing_path = BASE / "does_not_exist.txt"
print("exists?", missing_path.exists())


exists? False



### Mistake 4: mixing up `readline()` and `readlines()`

If you only want one line, do **not** call `readlines()`.



## 5. Useful helper functions for this unit


In [8]:

def read_text_lines(path):
    """Return file lines without trailing newline characters."""
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

def write_text_lines(path, lines):
    """Write a sequence of lines to a file, one line per item."""
    with open(path, "w", encoding="utf-8") as out:
        for line in lines:
            out.write(str(line) + "\n")



## 6. Worked example A — reverse the order of lines

### Problem
Read a text file and print its lines in reverse order.

This matches the first official exercise style for Unit 05. fileciteturn30file19


In [9]:

def reverse_lines_to_screen(path):
    lines = read_text_lines(path)
    return "\n".join(lines[::-1])

print(reverse_lines_to_screen(BASE / "simple_lines.txt"))


line3
line2
line1



## 7. Worked example B — reverse only nonblank lines and save to `reverse.txt`

This matches the second official exercise style. The book’s solution reverses the kept lines and writes the result to a file named `reverse.txt`. fileciteturn27file0turn30file19


In [10]:

def reverse_nonblank_lines_to_file(path, output_path="reverse.txt"):
    kept = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                kept.append(line.rstrip("\n"))

    out_path = Path(output_path)
    with open(out_path, "w", encoding="utf-8") as out:
        for line in reversed(kept):
            out.write(line + "\n")
    return out_path

reverse_path = reverse_nonblank_lines_to_file(BASE / "mixed_blank_lines.txt", BASE / "reverse.txt")
print(reverse_path.read_text(encoding="utf-8"))


gamma
beta
alpha




## 8. Worked example C — extract all headlines

The chapter includes a problem where each headline is on one line between `<headline>` and `</headline>`. fileciteturn30file19turn30file4


In [11]:

def extract_headlines(path):
    headlines = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            start = line.find("<headline>")
            if start >= 0:
                start += len("<headline>")
                end = line.find("</headline>", start)
                headlines.append(line[start:end])
    return headlines

extract_headlines(BASE / "news.txt")


['Rain expected tomorrow',
 'Stocks rise after report',
 'Python class cancelled']


## 9. Worked example D — compare two files

The chapter also includes comparing two files and printing `True` if they are identical, otherwise `False`. fileciteturn32file4


In [12]:

def same_file_contents(path1, path2):
    with open(path1, "r", encoding="utf-8") as f1, open(path2, "r", encoding="utf-8") as f2:
        return f1.readlines() == f2.readlines()

print(same_file_contents(BASE / "same1.txt", BASE / "same2.txt"))
print(same_file_contents(BASE / "same1.txt", BASE / "diff.txt"))


True
False



## 10. Worked example E — average score of a section

The chapter’s main example reads `data.txt`, where each line has the format:

```text
id:name:section:score
```

Then it computes the average score for a requested section, or prints `Not Found` if that section has no students. fileciteturn27file12


In [13]:

def average_score_by_section(path, wanted_section):
    total = 0.0
    count = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            sid, name, section, score = line.strip().split(":")
            if int(section) == wanted_section:
                total += float(score)
                count += 1
    if count == 0:
        return "Not Found"
    return total / count

print(average_score_by_section(BASE / "data.txt", 1))
print(average_score_by_section(BASE / "data.txt", 2))
print(average_score_by_section(BASE / "data.txt", 3))


56.6
88.66666666666667
Not Found



## 11. Clean file I/O style notes

When solving grader-style problems, you may see:

```python
f = open(filename)
...
f.close()
```

That works, but a cleaner version is:

```python
with open(filename, "r", encoding="utf-8") as f:
    ...
```

And for writing:

```python
with open(output_name, "w", encoding="utf-8") as out:
    ...
```

This notebook uses the cleaner form everywhere.



## 12. Practice problems

These practice tasks cover the official Unit 05 exercise style.

### Problem 1 — Reverse file lines to screen
Input: file name  
Process: read all lines and reverse their order  
Output: print reversed lines to the screen

### Problem 2 — Reverse only nonblank lines to `reverse.txt`
Input: file name  
Process: keep only nonblank lines, reverse them  
Output: save the result to `reverse.txt`

### Problem 3 — Extract headlines
Input: file name  
Process: find text between `<headline>` and `</headline>`  
Output: print one headline per line

### Problem 4 — Compare two files
Input: two file names  
Process: compare their contents exactly  
Output: `True` if identical, else `False`

### Problem 5 — Average score by section
Input: section number  
Process: read `data.txt`, compute average score of that section  
Output: average score or `Not Found`

### Problem 6 — Save primes less than `m` to a file
Input: integer `m`  
Process: find primes less than `m`, write them 5 per line  
Output: save to a text file

### Problem 7 — Clean name lookup
Input: target name  
Process: search for the exact name inside `names.txt`  
Output: found / not found, while handling newline cleanup correctly



## 13. Reference solutions


In [14]:

def solution1_reverse_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        lines = [line.rstrip("\n") for line in f]
    return "\n".join(reversed(lines))

def solution2_reverse_nonblank(path, output_path):
    with open(path, "r", encoding="utf-8") as f:
        kept = [line.rstrip("\n") for line in f if line.strip()]
    with open(output_path, "w", encoding="utf-8") as out:
        for line in reversed(kept):
            out.write(line + "\n")

def solution3_headlines(path):
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            a = line.find("<headline>")
            if a >= 0:
                j = a + len("<headline>")
                b = line.find("</headline>", j)
                out.append(line[j:b])
    return out

def solution4_compare_files(path1, path2):
    with open(path1, "r", encoding="utf-8") as f1, open(path2, "r", encoding="utf-8") as f2:
        return f1.readlines() == f2.readlines()

def solution5_average_section(path, wanted_section):
    total = 0.0
    count = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            sid, name, section, score = line.strip().split(":")
            if int(section) == wanted_section:
                total += float(score)
                count += 1
    return "Not Found" if count == 0 else total / count

def is_prime(n):
    if n < 2:
        return False
    for k in range(2, n):
        if n % k == 0:
            return False
    return True

def solution6_write_primes(m, output_path):
    count = 0
    buffer = []
    with open(output_path, "w", encoding="utf-8") as out:
        for n in range(2, m):
            if is_prime(n):
                buffer.append(str(n))
                count += 1
                if count % 5 == 0:
                    out.write(", ".join(buffer) + "\n")
                    buffer = []
        if buffer:
            out.write(", ".join(buffer) + "\n")

def solution7_name_lookup(path, target):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip() == target:
                return f"{target}: found in names.txt"
    return f"{target}: not found"



## 14. Self-checks

These assertions help verify that the notebook is working end to end.


In [15]:

assert solution1_reverse_lines(BASE / "simple_lines.txt") == "line3\nline2\nline1"

tmp_reverse = BASE / "reverse_check.txt"
solution2_reverse_nonblank(BASE / "mixed_blank_lines.txt", tmp_reverse)
assert tmp_reverse.read_text(encoding="utf-8") == "gamma\nbeta\nalpha\n"

assert solution3_headlines(BASE / "news.txt") == [
    "Rain expected tomorrow",
    "Stocks rise after report",
    "Python class cancelled",
]

assert solution4_compare_files(BASE / "same1.txt", BASE / "same2.txt") is True
assert solution4_compare_files(BASE / "same1.txt", BASE / "diff.txt") is False

assert solution5_average_section(BASE / "data.txt", 1) == 56.6
assert solution5_average_section(BASE / "data.txt", 2) == (78.0 + 89.0 + 99.0) / 3
assert solution5_average_section(BASE / "data.txt", 3) == "Not Found"

prime_out = BASE / "primes.txt"
solution6_write_primes(20, prime_out)
assert prime_out.read_text(encoding="utf-8") == "2, 3, 5, 7, 11\n13, 17, 19\n"

assert solution7_name_lookup(BASE / "names.txt", "Bob") == "Bob: found in names.txt"
assert solution7_name_lookup(BASE / "names.txt", "Daisy") == "Daisy: not found"

print("All Unit 05 checks passed.")


All Unit 05 checks passed.



## 15. Final notes

This unit is small in syntax, but very important in practice.

If you can do the following comfortably, you are in good shape:
- read a file line by line
- clean each line with `strip()` or `rstrip('\n')`
- split structured data into fields
- write results to a new file
- compare two files
- avoid newline and path mistakes

That is the core of Unit 05.
